# Count audit — NCHS neonatal population

Framework stage: **S4** (count audit) and gate **G1**, with the leakage check of gate **G2** for missingness. Reads `data/interim/cohort_<year>.parquet` (data adapter, notebook 01) and produces only aggregate counts in `data/outputs_02_count_audit.txt`; counts from 1 to 9 are shown as "<10". Frequencies and unknowns of the candidate predictors are computed in the whole population, never split by outcome; the only quantity that uses the outcome is the AUROC of the number of unknown fields alone (gate G2). Spanish version: `02_count_audit_es.ipynb` (same code). Takes a few minutes and about 3 GB of memory.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import COHORTS, OUT_DIR

OUT_FILE = ROOT / "data" / "outputs_02_count_audit.txt"
# Gate G1: reference counts of the data adapter (data/outputs_01_build_cohorts.txt):
# (population of 22-47 weeks, neonatal deaths 0-27 days)
EXPECTED = {2016: (3938288, 11497), 2017: (3848558, 11057), 2018: (3785243, 10703),
            2019: (3741187, 10249), 2020: (3607479, 9574), 2021: (3657762, 9633),
            2022: (3661524, 9967), 2023: (3589826, 9879)}
# "Unknown or not stated" codes (user guides). Character fields use "U"; DMAR is blank when the
# state does not report marital status. "Not applicable" codes (ILP_R 888, ME_TRIAL "X") are
# categories, not unknowns.
UNKNOWN = {"MRACEHISP": [8], "MEDUC": [9], "MBSTATE_REC": [3], "PAY_REC": [9], "BFACIL3": [9],
           "PRIORLIVE": [99], "PRIORDEAD": [99], "PRIORTERM": [99], "ILP_R": [999],
           "RF_CESARN": [99], "PRECARE": [99], "PREVIS": [99], "BMI": [99.9], "WTGAIN": [99],
           "ME_PRES": [9], "ME_ROUT": [9], "ATTEND": [9], "APGAR5": [99], "BRTHWGT": [9999]}
CHAR_UNKNOWN = "U"
BLOCKS = {
    "mother": ["MAGER", "MRACEHISP", "MEDUC", "DMAR", "MBSTATE_REC", "PAY_REC", "WIC"],
    "obstetric_history": ["PRIORLIVE", "PRIORDEAD", "PRIORTERM", "ILP_R", "RF_CESAR", "RF_CESARN", "RF_PPB"],
    "current_pregnancy": ["PRECARE", "PREVIS", "CIG_REC", "BMI", "WTGAIN", "RF_PDIAB", "RF_GDIAB",
                          "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_INFT", "IP_GON", "IP_SYPH",
                          "IP_CHLAM", "IP_HEPB", "IP_HEPC"],
    "labor_delivery": ["LD_INDL", "LD_AUGM", "LD_STER", "LD_ANTB", "LD_CHOR", "LD_ANES", "ME_PRES",
                       "ME_ROUT", "ME_TRIAL", "BFACIL3", "ATTEND"],
    "newborn": ["DPLURAL", "SEX", "COMBGEST", "BRTHWGT", "APGAR5", "CA_ANEN", "CA_MNSB", "CA_CCHD",
                "CA_CDH", "CA_OMPH", "CA_GAST", "CA_LIMB", "CA_CLEFT", "CA_CLPAL", "CA_DOWN",
                "CA_DISOR", "CA_HYPO"],
}
FIELDS = [c for b in BLOCKS.values() for c in b]
NUMERIC = ["MAGER", "PRIORLIVE", "PRIORDEAD", "PRIORTERM", "ILP_R", "RF_CESARN", "PRECARE", "PREVIS",
           "BMI", "WTGAIN", "COMBGEST", "BRTHWGT", "APGAR5"]
CATEGORICAL = [c for c in FIELDS if c not in NUMERIC]
LABELS = ["label_main", "died_first_24h", "label_day1_27", "label_early_neonatal"]
DEATH = ["died_infant", "d_age_days", "d_ager5"]
lines = []

def show(title, obj):
    """Print and keep an aggregate table; counts from 1 to 9 are suppressed as '<10'."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        sup = lambda v: ("<10" if isinstance(v, (int, np.integer)) and not isinstance(v, (bool, np.bool_))
                         and 0 < v < 10 else v)
        txt = (obj.map(sup) if isinstance(obj, pd.DataFrame) else obj.map(sup)).to_string()
    else:
        txt = str(obj)
    block = f"\n== {title}\n{txt}"
    print(block)
    lines.append(block)

def is_unknown(s, field):
    """True where the field is unknown or not stated (blank counts as unknown)."""
    if field == "BMI":                       # stored as float: compare with a tolerance
        return s.isna() | (s.astype("Float64") > 99.85).fillna(False)
    if field in UNKNOWN:
        return s.isna() | s.isin(UNKNOWN[field])
    if isinstance(s.dtype, pd.CategoricalDtype) or s.dtype == object:
        return s.isna() | (s.astype("string") == CHAR_UNKNOWN)
    return s.isna()

print("cohorts:", COHORTS)

## 1. Population, events by label and gate G1

In [ ]:
# Q-N1 population and events by label; Q-N2 age at death in days; Q-N3 gestational-age bands;
# accumulators for Q-N4 to Q-N7 (whole population, never split by outcome)
pop_rows, age_rows, ga_rows, unk_year = [], [], [], []
cat_counts = {c: {} for c in CATEGORICAL}
num_counts = {c: {} for c in NUMERIC}
nunk_main, nunk_ge28 = {}, {}          # number of unknowns -> [records, neonatal deaths] (gate G2)
imputed = []
GA_BANDS = [(22, 23), (24, 27), (28, 31), (32, 36), (37, 47)]
for y in COHORTS:
    path = OUT_DIR / f"cohort_{y}.parquet"
    d = pd.read_parquet(path, columns=["RESTATUS", "COMBGEST", "COMBGST_IMP", "BWTIMP"] + LABELS + DEATH)
    keep = ((d["RESTATUS"] < 4) & d["COMBGEST"].between(22, 47)).to_numpy()
    d = d[keep]
    ge28 = d["COMBGEST"] >= 28
    pop_rows.append({"cohort": y, "population": len(d), "events_0_27d": int(d["label_main"].sum()),
                     "first_24h": int(d["died_first_24h"].sum()), "day1_27": int(d["label_day1_27"].sum()),
                     "early_0_6d": int(d["label_early_neonatal"].sum()),
                     "post_neonatal_28d_1y": int((d["died_infant"] & ~d["label_main"]).sum()),
                     "population_ge28": int(ge28.sum()), "events_ge28": int((ge28 & d["label_main"]).sum()),
                     "per_1000": round(1000 * d["label_main"].mean(), 2)})
    dead = d[d["label_main"]]
    a = dead["d_age_days"].astype("Int64")
    row = {"cohort": y, "under_1h": int((dead["d_ager5"] == 1).sum()), "1_23h": int((dead["d_ager5"] == 2).sum())}
    row.update({f"day_{k}": int((a == k).sum()) for k in range(1, 7)})
    row["day_7_27"] = int(a.between(7, 27).sum())
    row["age_missing"] = int(a.isna().sum())
    age_rows.append(row)
    for lo, hi in GA_BANDS:
        g = d["COMBGEST"].between(lo, hi)
        ga_rows.append({"cohort": y, "band": f"{lo}-{hi}", "population": int(g.sum()),
                        "events_0_27d": int((g & d["label_main"]).sum()),
                        "first_24h": int((g & d["died_first_24h"]).sum())})
    imputed.append({"cohort": y, "ga_imputed_pct": round(100 * (d["COMBGST_IMP"] == "1").mean(), 2),
                    "bw_imputed_pct": round(100 * (d["BWTIMP"] == "1").mean(), 2)})
    label = d["label_main"].to_numpy(bool)
    ge28 = ge28.to_numpy()
    del d, dead
    # one field at a time keeps the memory low (about 1 GB)
    n = np.zeros(int(keep.sum()), dtype=np.int16)
    row = {"cohort": y}
    for c in FIELDS:
        s = pd.read_parquet(path, columns=[c])[c][keep].reset_index(drop=True)
        u = is_unknown(s, c).to_numpy(bool)
        n += u
        row[c] = round(100 * u.mean(), 2)
        if c in CATEGORICAL:
            for k, v in s.astype("object").where(s.notna(), "missing").value_counts().items():
                cat_counts[c][k] = cat_counts[c].get(k, 0) + int(v)
        else:
            for k, v in s.astype("Float64").fillna(-1).value_counts().items():
                num_counts[c][round(float(k), 1)] = num_counts[c].get(round(float(k), 1), 0) + int(v)
        del s, u
    unk_year.append(row)
    for store, mask in ((nunk_main, np.ones(len(n), bool)), (nunk_ge28, ge28)):
        recs = np.bincount(n[mask]); evs = np.bincount(n[mask], weights=label[mask], minlength=len(recs))
        for k in np.nonzero(recs)[0]:
            r = store.setdefault(int(k), [0, 0])
            r[0] += int(recs[k]); r[1] += int(evs[k])
    del n, label, ge28, keep
    print(y, "done")

pop = pd.DataFrame(pop_rows).set_index("cohort")
show("Q-N1 Population (US residents, 22-47 weeks) and events by label", pop)
g1 = pd.Series({y: (int(pop.loc[y, "population"]), int(pop.loc[y, "events_0_27d"])) == EXPECTED[y]
                for y in COHORTS}, name="reproduced")
show("Gate G1: reference counts reproduced", g1)
assert g1.all(), "gate G1: the counts do not match the data adapter"

## 2. Age at death and gestational-age bands

The 24-hour landmark (sensitivity a) depends on how the age at death is coded on day 0 and day 1; this table checks that the coding is stable across cohorts.

In [ ]:
# Q-N2 age at death of the neonatal deaths: hours on day 0, then days
age = pd.DataFrame(age_rows).set_index("cohort")
show("Q-N2 Neonatal deaths by age at death (day 0 split into <1 h and 1-23 h)", age)
share = pd.DataFrame({"day0_pct": (100 * (age["under_1h"] + age["1_23h"]) / pop["events_0_27d"]).round(1),
                      "day1_pct": (100 * age["day_1"] / pop["events_0_27d"]).round(1),
                      "day0_plus_day1_pct": (100 * (age["under_1h"] + age["1_23h"] + age["day_1"])
                                             / pop["events_0_27d"]).round(1),
                      "day2_27_pct": (100 * (pop["events_0_27d"] - age["under_1h"] - age["1_23h"] - age["day_1"])
                                      / pop["events_0_27d"]).round(1)})
show("Q-N2b Share of the neonatal deaths: day 0, day 1, days 0-1 together and days 2-27", share)

# Q-N3 gestational-age bands, all cohorts together
ga = pd.DataFrame(ga_rows).groupby("band", sort=False)[["population", "events_0_27d", "first_24h"]].sum()
ga["pct_population"] = (100 * ga["population"] / ga["population"].sum()).round(2)
ga["pct_events"] = (100 * ga["events_0_27d"] / ga["events_0_27d"].sum()).round(1)
ga["per_1000"] = (1000 * ga["events_0_27d"] / ga["population"]).round(1)
show("Q-N3 Gestational-age bands (combined gestation), cohorts 2016-2023", ga)
show("Q-N3b Gestational age and birthweight imputed by NCHS (% of the population)",
     pd.DataFrame(imputed).set_index("cohort"))

## 3. Frequencies of the candidate predictors

In [ ]:
# Q-N4 frequencies of the categorical candidates (whole population, never split by outcome)
total = int(pop["population"].sum())
for block, cols in BLOCKS.items():
    for c in cols:
        if c in CATEGORICAL:
            s = pd.Series(cat_counts[c])
            s = s.reindex(sorted(s.index, key=str))
            show(f"Q-N4 [{block}] {c}: births and % of the population",
                 pd.DataFrame({"n": s, "pct": (100 * s / total).round(2)}))

# Q-N5 numeric candidates: unknown and percentiles (unknown codes and the 888 'not applicable' excluded)
rows = []
for c in NUMERIC:
    s = pd.Series(num_counts[c])
    excl = set(UNKNOWN.get(c, [])) | {-1.0} | ({888.0} if c == "ILP_R" else set())
    unk = s[[k for k in s.index if k in excl and k != 888.0]].sum()
    v = s[[k for k in s.index if k not in excl and not (c == "BMI" and abs(k - 99.9) < 1e-3)]].sort_index()
    cum = v.cumsum() / v.sum()
    row = {"field": c, "pct_unknown": round(100 * unk / total, 2)}
    row.update({f"p{int(q * 100):02d}": float(v.index[np.searchsorted(cum.values, q)])
                for q in (0.05, 0.25, 0.5, 0.75, 0.95)})
    if c == "ILP_R":
        row["pct_not_applicable_888"] = round(100 * s.get(888.0, 0) / total, 2)
    rows.append(row)
show("Q-N5 Numeric candidates: % unknown and percentiles", pd.DataFrame(rows).set_index("field"))

## 4. Unknowns by cohort, leakage check (gate G2) and aggregate output

In [ ]:
# Q-N6 % unknown of every candidate by cohort (whole population)
uy = pd.DataFrame(unk_year).set_index("cohort").T
uy["all_cohorts"] = (uy.mul(pop["population"], axis=1).sum(axis=1) / total).round(2)
show("Q-N6 % unknown by cohort, whole population", uy)
show("Q-N6b Candidates with more than 5% unknown in any cohort", uy[uy.drop(columns="all_cohorts").max(axis=1) > 5])

# Q-N7 leakage check (gate G2): does the number of unknown fields, by itself, separate the deaths?
def auroc_from_counts(table):
    """AUROC of an integer score from a {score: [records, events]} table (ties count one half)."""
    t = pd.DataFrame(table, index=["records", "events"]).T.sort_index()
    ev = t["events"].to_numpy(float); ne = (t["records"] - t["events"]).to_numpy(float)
    below = np.concatenate([[0], np.cumsum(ne)[:-1]])
    return float((ev * (below + 0.5 * ne)).sum() / (ev.sum() * ne.sum()))

def summary(table):
    t = pd.DataFrame(table, index=["records", "events"]).T.sort_index()
    t.index = [str(k) if k < 5 else "5+" for k in t.index]
    return t.groupby(level=0, sort=False).sum()

g2 = pd.DataFrame({"population": [auroc_from_counts(nunk_main)], "ge28_weeks": [auroc_from_counts(nunk_ge28)]},
                  index=["AUROC of the number of unknowns"]).round(3)
show("Q-N7 Gate G2: AUROC of the number of unknown fields alone (neonatal death 0-27 days)", g2)
show("Q-N7b Number of unknown fields: births (whole population)", summary(nunk_main)[["records"]])

OUT_FILE.write_text("Count audit of the neonatal population (aggregate counts only; 1-9 shown as <10)\n"
                    + "\n".join(lines) + "\n", encoding="utf-8")
print("written:", OUT_FILE)